<a href="https://colab.research.google.com/github/akshaybhatt97/365-DAYS-AI-ML-WITH-AKSHAY/blob/main/Day_107_Vector_Search_Practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 107/365 — Vector Databases & Nearest-Neighbor Search

Yesterday:

`Text → Vector → Cosine Similarity`

Today:

`Documents → Vectors → Index → Query → Top-K Results`

We use real SMS text from the **UCI SMS Spam Collection**.

This notebook implements an **exact vector index** with scikit-learn's `NearestNeighbors`.

A production vector database adds persistence, metadata filtering, scalable indexing, CRUD/upserts, replication and operational tooling around the same retrieval idea.


In [1]:
# If needed:
# !pip install pandas scikit-learn matplotlib

import io
import zipfile
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA


## 1. Load the real UCI SMS Spam Collection

In [2]:
UCI_URL = (
    "https://archive.ics.uci.edu/static/public/228/"
    "sms%2Bspam%2Bcollection.zip"
)

fallback_rows = [
    ("ham", "Ok lar... Joking wif u oni..."),
    ("ham", "Nah I don't think he goes to usf, he lives around here though"),
    ("ham", "Sorry, I'll call later in meeting."),
    ("ham", "I'm home."),
    ("ham", "Can you call me when you reach home"),
    ("ham", "I will be late today. Meeting is still going on"),
    ("ham", "Please call me after your meeting"),
    ("ham", "I reached home safely"),
    ("ham", "Call me once your meeting finishes"),
    ("ham", "Let me know when you get home"),
    ("spam", "WINNER!! As a valued network customer you have been selected to receive a prize reward!"),
    ("spam", "URGENT! You have won a FREE membership in our Prize Jackpot! Txt CLAIM now."),
    ("spam", "Congratulations! You have won a cash prize. Reply CLAIM now."),
    ("spam", "Free offer! Text WIN now to claim your reward."),
    ("spam", "Claim your free prize now. Text WIN to receive your reward."),
]

try:
    raw = urllib.request.urlopen(UCI_URL, timeout=20).read()

    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        with z.open("SMSSpamCollection") as f:
            df = pd.read_csv(
                f,
                sep="\\t",
                header=None,
                names=["label","text"]
            )

    source = "Full UCI SMS Spam Collection"

except Exception:
    df = pd.DataFrame(
        fallback_rows,
        columns=["label","text"]
    )
    source = "Offline real-data fallback sample"

df["id"] = [
    f"sms-{i:05d}"
    for i in range(len(df))
]

print("Source:", source)
print("Rows:", len(df))
df.head()


/tmp/ipykernel_17533/1196221591.py:29: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv(


Source: Full UCI SMS Spam Collection
Rows: 5574


,label,text,id
0,ham,"Go until jurong point, crazy.. Available only ...",sms-00000
1,ham,Ok lar... Joking wif u oni...,sms-00001
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,sms-00002
3,ham,U dun say so early hor... U c already then say...,sms-00003
4,ham,"Nah I don't think he goes to usf, he lives aro...",sms-00004


## 2. What does a vector record contain?

Conceptually, a vector database stores something like:

```python
{
    "id": "sms-00042",
    "vector": [...],
    "metadata": {"label": "ham"},
    "text": "Please call me after your meeting"
}
```

The vector is searched.

The ID, text and metadata are returned with the result.


## 3. Convert real text into vectors

In [3]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1,2),
    stop_words="english"
)

document_vectors = (
    vectorizer.fit_transform(
        df["text"]
    )
)

print(
    "Documents:",
    document_vectors.shape[0]
)

print(
    "Vector dimensions:",
    document_vectors.shape[1]
)


Documents: 5574
Vector dimensions: 37360


TF-IDF is used here because it runs locally and makes the indexing flow easy to inspect.

A production semantic-search system can replace these vectors with neural embeddings without changing the overall retrieval pipeline.


## 4. Build an exact nearest-neighbor index

In [4]:
index = NearestNeighbors(
    metric="cosine",
    algorithm="brute"
)

index.fit(
    document_vectors
)


NearestNeighbors(algorithm='brute', metric='cosine')

## 5. Query the index

In [5]:
query = (
    "please call me when "
    "the meeting is over"
)

query_vector = (
    vectorizer.transform(
        [query]
    )
)

K = 5

distances, indices = (
    index.kneighbors(
        query_vector,
        n_neighbors=min(
            K,
            len(df)
        )
    )
)

similarities = (
    1 - distances[0]
)

result_indices = (
    indices[0]
)

results = df.iloc[
    result_indices
].copy()

results["similarity"] = (
    similarities
)

results[
    [
        "id",
        "label",
        "text",
        "similarity"
    ]
]


,id,label,text,similarity
1093,sms-01093,ham,Where r we meeting?,1.000000
3178,sms-03178,ham,In meeting da. I will call you,0.531784
3392,sms-03392,ham,"I'm in a meeting, call me later at",0.531473
4633,sms-04633,ham,"I'm in a meeting, call me later at",0.531473
590,sms-00590,ham,"I'm in a meeting, call me later at",0.531473


This is **Top-K nearest-neighbor search**.

`K = 5` means:

return the five closest stored vectors.


## 6. Build a reusable search function

In [6]:
def search(
    query,
    k=5,
    label_filter=None
):

    query_vector = (
        vectorizer.transform(
            [query]
        )
    )

    candidate_count = min(
        len(df),
        max(k * 5, k)
    )

    distances, indices = (
        index.kneighbors(
            query_vector,
            n_neighbors=candidate_count
        )
    )

    similarities = (
        1 - distances[0]
    )

    rows = []

    for idx, score in zip(
        indices[0],
        similarities
    ):

        record = df.iloc[idx]

        if (
            label_filter is not None
            and record["label"]
            != label_filter
        ):
            continue

        rows.append({
            "id": record["id"],
            "label": record["label"],
            "text": record["text"],
            "similarity": float(score)
        })

        if len(rows) == k:
            break

    return pd.DataFrame(rows)


In [7]:
search(
    "claim my free reward",
    k=5
)


,id,label,text,similarity
0,sms-04419,ham,"When you get free, call me",0.334343
1,sms-03364,ham,Can... I'm free...,0.334343
2,sms-00495,ham,Are you free now?can i call now?,0.334343
3,sms-05192,spam,Our records indicate u maybe entitled to 5000 ...,0.237979
4,sms-04236,spam,FREEMSG: Our records indicate you may be entit...,0.202870


## 7. Metadata filtering

In [8]:
search(
    "claim my free reward",
    k=5,
    label_filter="spam"
)


,id,label,text,similarity
0,sms-05192,spam,Our records indicate u maybe entitled to 5000 ...,0.237979
1,sms-04236,spam,FREEMSG: Our records indicate you may be entit...,0.202870
2,sms-05368,spam,A £400 XMAS REWARD IS WAITING FOR YOU! Our com...,0.190663
3,sms-04652,spam,A £400 XMAS REWARD IS WAITING FOR YOU! Our com...,0.190663
4,sms-04281,spam,WINNER!! As a valued network customer you have...,0.185934


Real vector databases commonly combine:

**vector similarity + metadata filters**

Useful metadata may include:

- customer ID
- document type
- department
- date
- access-control scope
- product
- tenant


## 8. Why Top-K matters

In [9]:
for k in [1,3,5]:

    print(
        "\nTOP",
        k
    )

    print(
        search(
            "call after meeting",
            k=k
        )[
            ["text","similarity"]
        ]
    )



TOP 1
                  text  similarity
0  Where r we meeting?         1.0

TOP 3
                                 text  similarity
0                 Where r we meeting?    1.000000
1      In meeting da. I will call you    0.531784
2  I'm in a meeting, call me later at    0.531473

TOP 5
                                 text  similarity
0                 Where r we meeting?    1.000000
1      In meeting da. I will call you    0.531784
2  I'm in a meeting, call me later at    0.531473
3  I'm in a meeting, call me later at    0.531473
4  I'm in a meeting, call me later at    0.531473


In RAG, `K` affects:

- recall
- prompt size
- latency
- token cost
- irrelevant context

Larger `K` is not automatically better.


## 9. Exact vs Approximate Nearest Neighbor search

Our index is exact.

For a small dataset, brute-force comparison is fine.

At large scale:

`100 million vectors × 1,536 dimensions`

exact comparison becomes expensive.

Production vector systems often use **Approximate Nearest Neighbor (ANN)** indexes.

They trade a small amount of exactness for much faster retrieval.

Popular ideas include:

- HNSW
- IVF
- product quantization


## 10. Vector database vs ordinary database

Traditional database query:

`WHERE user_id = 123`

Vector-search query:

> Find the stored records closest in meaning to this query vector.

Modern systems often combine both:

`metadata filters + vector similarity`


## 11. Connection to RAG

### Ingestion

`Documents`

→ chunk text

→ generate embeddings

→ store:

`vector + text + metadata`

### Retrieval

`User Question`

→ query embedding

→ Top-K nearest-neighbor search

→ relevant chunks

→ LLM prompt

This is the retriever at the heart of many RAG systems.


## Simple takeaway

A vector index turns:

`query vector`

into:

`Top-K nearest stored vectors`

A vector database surrounds this operation with storage, filtering, persistence and scalable indexes.

> **Embeddings represent meaning. Nearest-neighbor search retrieves the closest representations.**


## References

- UCI SMS Spam Collection:
  https://archive.ics.uci.edu/dataset/228/sms+spam+collection

- scikit-learn NearestNeighbors:
  https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.NearestNeighbors.html

- FAISS:
  https://github.com/facebookresearch/faiss

- HNSW paper:
  https://arxiv.org/abs/1603.09320
